In [1]:
#packages 
import pandas as pd

In [2]:
# Use the manually installed source CSVs; this notebook makes no API requests.
imdb = pd.read_csv("data/imdb_movies.csv", dtype={"imdb_id": "string", "nconst": "string"}, na_values=[r"\N"])
tmdb = pd.read_csv("data/tmdb_movies.csv", dtype={"imdb_id": "string"}, na_values=[r"\N"])
required_tmdb = {"tmdb_id", "imdb_id", "title", "director", "collection_id", "collection_name", "us_certification"}
required_imdb = {"imdb_id", "nconst", "title", "director", "startYear"}
for label, dataset, required in [("TMDB", tmdb, required_tmdb), ("IMDb", imdb, required_imdb)]:
    missing = required - set(dataset.columns)
    if missing:
        raise ValueError(f"{label} is missing expected new-schema columns: {missing}")
    dataset["imdb_id"] = dataset["imdb_id"].str.strip().replace("", pd.NA)
imdb["nconst"] = imdb["nconst"].str.strip().replace("", pd.NA)
imdb["startYear"] = pd.to_numeric(imdb["startYear"], errors="coerce").astype("Int64")
tmdb["collection_id"] = pd.to_numeric(tmdb["collection_id"], errors="coerce").astype("Int64")
print(f"Loaded local sources: {len(tmdb):,} TMDB movies; {len(imdb):,} IMDb movie–director rows")


Loaded local sources: 14,509 TMDB movies; 764,246 IMDb movie–director rows


In [3]:
# Multiple IMDb rows per movie are expected when it has co-directors.
print("Missing movie IDs (TMDB, IMDb):", tmdb["imdb_id"].isna().sum(), imdb["imdb_id"].isna().sum())
print("Repeated movie IDs (TMDB, IMDb):", tmdb["imdb_id"].duplicated().sum(), imdb["imdb_id"].duplicated().sum())


Missing movie IDs (TMDB, IMDb): 10 0
Repeated movie IDs (TMDB, IMDb): 9 84558


Missing IDs cannot be joined. Repeated IMDb movie IDs are expected for co-directed films; duplicate movie–director pairs are checked separately. Counts below are calculated from the current local sources.


In [4]:
# An inner join needs movie IDs in both sources.
tmdb = tmdb.dropna(subset=["imdb_id"])
imdb = imdb.dropna(subset=["imdb_id", "nconst"])
imdb = imdb.drop_duplicates(subset=["imdb_id", "nconst"])
if not tmdb["imdb_id"].is_unique:
    raise ValueError("TMDB has repeated IMDb movie IDs; resolve them before joining.")
print("Unique movie–director pairs:", len(imdb))


Unique movie–director pairs: 764246


In [5]:
# Keep source labels explicit. Output has one row per matched movie–director pair.
combined = tmdb.merge(
    imdb, on="imdb_id", how="inner",
    suffixes=("_tmdb", "_imdb"), validate="one_to_many",
)
print("TMDB Shape:", tmdb.shape)
print("IMDb Shape:", imdb.shape)
print("combined shapes:", combined.shape)
print("Matched unique movies:", combined["tmdb_id"].nunique())
print("^Because The inner join drops movies without an IMDb match and \na movie with two directors gets two rows, one for each director.")
combined.head()


TMDB Shape: (14499, 18)
IMDb Shape: (764246, 5)
combined shapes: (14824, 22)
Matched unique movies: 13276
^Because The inner join drops movies without an IMDb match and 
a movie with two directors gets two rows, one for each director.


,tmdb_id,imdb_id,title_tmdb,release_date,budget,revenue,runtime,genres,original_language,popularity,...,top_cast,vote_count,vote_average,collection_id,collection_name,us_certification,nconst,director_imdb,title_imdb,startYear
0,769,tt0099685,GoodFellas,1990-09-12,25000000,47072327,145,Drama|Crime,en,32.6399,...,Robert De Niro|Ray Liotta|Joe Pesci|Lorraine B...,14953,8.451,<NA>,NaN,R,nm0000217,Martin Scorsese,GoodFellas,1990
1,162,tt0099487,Edward Scissorhands,1990-12-07,20000000,86024005,105,Fantasy|Drama|Romance,en,17.6934,...,Johnny Depp|Winona Ryder|Dianne Wiest|Anthony ...,14152,7.719,<NA>,NaN,PG-13,nm0000318,Tim Burton,Edward Scissorhands,1990
2,771,tt0099785,Home Alone,1990-11-16,18000000,476684675,103,Comedy|Family,en,1.7507,...,Macaulay Culkin|Joe Pesci|Daniel Stern|John He...,12999,7.470,9888,Home Alone Collection,PG,nm0001060,Chris Columbus,Home Alone,1990
3,196,tt0099088,Back to the Future Part III,1990-05-25,40000000,245077583,119,Adventure|Comedy|Science Fiction,en,15.8103,...,Michael J. Fox|Christopher Lloyd|Mary Steenbur...,11962,7.495,264,Back to the Future Collection,PG,nm0000709,Robert Zemeckis,Back to the Future Part III,1990
4,114,tt0100405,Pretty Woman,1990-03-23,14000000,463406268,120,Romance|Comedy,en,17.7921,...,Julia Roberts|Richard Gere|Ralph Bellamy|Jason...,9275,7.441,<NA>,NaN,R,nm0005190,Garry Marshall,Pretty Woman,1990


In [6]:
# Inspect Combined Column Names
print(combined.columns.tolist())

['tmdb_id', 'imdb_id', 'title_tmdb', 'release_date', 'budget', 'revenue', 'runtime', 'genres', 'original_language', 'popularity', 'production_companies', 'director_tmdb', 'top_cast', 'vote_count', 'vote_average', 'collection_id', 'collection_name', 'us_certification', 'nconst', 'director_imdb', 'title_imdb', 'startYear']


In [7]:
# Compare Titles Across the Two Sources
print((combined["title_tmdb"] == combined["title_imdb"]).mean())

0.9211413923367512


The title agreement fraction measures exact text agreement among joined rows, not join correctness. Records are matched by IMDb ID; translated titles or spelling differences can still be valid matches.


In [8]:
# Export derived data only; keep either source CSV unchanged.
from pathlib import Path
output_path = Path("data/combined_dataset.csv")
if output_path.exists():
    existing_output = pd.read_csv(output_path, dtype={"imdb_id": "string", "nconst": "string", "collection_id": "Int64", "startYear": "Int64"})
    try:
        pd.testing.assert_frame_equal(existing_output, combined.reset_index(drop=True), check_dtype=False, rtol=1e-12, atol=1e-12)
    except AssertionError:
        raise FileExistsError(f"{output_path} has different data. Move or rename it before exporting.") from None
    print(f"Existing {output_path} matches this run; left unchanged.")
else:
    with output_path.open("x") as output_file:
        combined.to_csv(output_file, index=False)
    print(f"Created {output_path}: {len(combined):,} rows and {len(combined.columns)} columns")


Existing data/combined_dataset.csv matches this run; left unchanged.
